## Imports

In [ ]:
import pandas as pd
import requests, io


url_datagouv_xslt = 'https://www.data.gouv.fr/api/1/datasets/r/6d9b33e5-667d-4c3e-9a0b-5fdf5baac708'

try:
	response = requests.get(url_datagouv_xslt)
	response.raise_for_status()

	df = pd.read_excel(io.BytesIO(response.content))
except requests.exceptions.HTTPError as http_err:
	print(f"Erreur HTTP détectée : {http_err}")
except Exception as e:
	print(f"Error: {str(e)}")
	

display(df)

In [ ]:
duplicates_count = df.duplicated().sum()
print('Nombre de doublons :', duplicates_count)

null_values_count = df.isnull().sum()
print(f'\nNombre de valeurs manquantes : {null_values_count}')

print('\nColonnes :')
for c, i in enumerate(df.columns):
    print(f'   {i}- {c}')

## Structuration des données

On veut :
- conserver les colonnes fixes et utiles (Inscrits, Abstentions, etc.)
- transformer le code de département + code de la commune en un code INSEE à 5 chiffres
- ajouter une colonne année pour anticiper la récupération d'autres élections
- convertir les colonnes Unnamed (contenant un candidat et le nombre de voix pour ce candidat) en nombre de voix pour la droite, la gauche, le centre

In [ ]:
# On convertit les codes département et commune en code INSEE + on ajoute une colonne année
df['Code_INSEE'] = df['Code du département'].astype(str) + df['Code de la commune'].astype(str).str.zfill(3)
df['Année'] = 2022

# On garde les colonnes nommées utiles
named_cols = ['Année', 'Code_INSEE', 'Libellé de la commune', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot']

df_clean = df[named_cols].copy()

display(df_clean)

In [ ]:
# On convertit les colonnes Unnamed en nombre de voix par candidat
# On commence à la première colonne N°Panneau et on traite 7 colonnes par candidat
start_index = df.columns.get_loc('N°Panneau')
last_index = len(df.columns) - 1

df_votes = df_clean[['Code_INSEE', 'Exprimés']].copy()

for i in range(int((last_index - start_index) / 7)):
    name_index = start_index + 2 + (i * 7)
    votes_index = start_index + 4 + (i * 7)
    name_value = df.iloc[0, name_index]
    df_votes[f'Voix_{name_value}'] = df.iloc[:, votes_index].astype(int)


display(df_votes)


In [ ]:
# On transforme les voix par candidat en voix par groupe politique (gauche, droite, centre)

groups = {
    'gauche': ['Voix_ARTHAUD', 'Voix_ROUSSEL', 'Voix_MÉLENCHON', 'Voix_HIDALGO', 'Voix_JADOT', 'Voix_POUTOU'],
    'centre': ['Voix_MACRON', 'Voix_LASSALLE'],
    'droite': ['Voix_LE PEN', 'Voix_ZEMMOUR', 'Voix_PÉCRESSE', 'Voix_DUPONT-AIGNAN']
}


for group, candidates in groups.items():
    total_votes = df_votes[candidates].sum(axis=1)
    df_clean[f'% {group}/Exp'] = (total_votes / df_votes['Exprimés'] * 100).round(2)


# On vérifie les valeurs manquantes : certaines communes ont 0 vote exprimé, donc pas de pourcentages par groupe politique
null_values_count = df_clean.isnull().sum()
print('\nNombre de valeurs manquantes :', null_values_count)


In [ ]:
# On ajoute le groupe majoritaire en tenant compte des valeurs manquantes
cols_groups = ['% gauche/Exp', '% centre/Exp', '% droite/Exp']
mask = df_clean[cols_groups].notna().any(axis=1)
df_clean.loc[mask, 'Résultat'] = df_clean.loc[mask, cols_groups].idxmax(axis=1)

groups_dict = {'% gauche/Exp': 'gauche', '% centre/Exp': "centre", '% droite/Exp': 'droite'}
df_clean['Résultat'] = df_clean['Résultat'].replace(groups_dict)


display(df_clean)

## Export du dataframe

In [ ]:
from pathlib import Path

if Path('src/election.csv').exists():
    print('Le fichier existe déjà, supprimez-le pour le mettre à jour')
else:
    df_clean.to_csv('src/election.csv', index=False, sep=';', encoding='utf-8')